# FedGrid-RL on Colab

1. **Runtime > Change runtime type > T4 GPU.** Without a GPU, Colab is slower than the laptop.
2. Upload `fedgrid_colab.zip` (code, processed data, feeder L and the results so far) to `My Drive/fedgrid/`.
3. **Runtime > Run all.**

Logs are written to `My Drive/fedgrid/results/`. If Colab disconnects, run all again: finished runs are skipped.
Afterwards, download the new logs from that folder into `results/` on the laptop.

In [ ]:
!nvidia-smi -L  # must list a GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
DRIVE=/content/drive/MyDrive/fedgrid
rm -rf /content/fedgrid && mkdir -p /content/fedgrid && cd /content/fedgrid
unzip -q $DRIVE/fedgrid_colab.zip
pip install -q opendssdirect.py scikit-learn
# Logs live on Drive so they survive a disconnect. The zip's logs are copied there first, never overwriting.
mkdir -p $DRIVE/results && cp -rn results/. $DRIVE/results/ && rm -rf results && ln -s $DRIVE/results results
ls results/*.log | wc -l

In [ ]:
# Speed check: 25 days of FedAvg. The laptop takes 85 s for this on its own, and 4 runs at once slow each to
# about 80 minutes per 500 day run. Well under 85 s here means Colab is worth it.
%cd /content/fedgrid
!OMP_NUM_THREADS=1 python src/agents/federated.py fedavg 25 0 | grep trained

In [ ]:
# Every attack run not yet in results/, 2 at a time (Colab has 2 CPU cores for the street simulation)
%cd /content/fedgrid
!JOBS=2 PYTHON=python ./run_experiments.sh attacks